In [1]:
import pandas as pd
from pathlib import Path

In [6]:
StartPath = Path.cwd().parents[0]
_FULL_PROCESSED_FILEPATH = StartPath / "data" / "full_preprocessed_dataset.csv"
_CHUNKSIZE = 10000

reader = pd.read_csv(
    _FULL_PROCESSED_FILEPATH,
    chunksize=_CHUNKSIZE,
    quotechar='"',
    usecols=['type'],
    low_memory=False
)
reliable_count = 0
non_reliable_count = 0
reliable_point = 0
non_reliable_point = 0

for i, chunk in enumerate(reader, 1):
    print(i)
    reliable_count_chunk = (chunk['type'] == 'reliable').sum()
    reliable_count += reliable_count_chunk
    non_reliable_count_chunk = (chunk['type'] != 'reliable').sum()
    non_reliable_count += non_reliable_count_chunk
    if non_reliable_count_chunk > reliable_count_chunk:
        non_reliable_point = i
    elif non_reliable_count_chunk < reliable_count_chunk and reliable_point == 0:
        reliable_point = i
    print(f'Reliable count: {reliable_count}')
    print(f'Non reliable count: {non_reliable_count}')
    print(f'Reliable ratio to total: {reliable_count/(reliable_count+non_reliable_count)}')

print(f'Reliable count: {reliable_count}')
print(f'Non reliable count: {non_reliable_count}')
print(f'Reliable ratio to total: {reliable_count/(reliable_count+non_reliable_count)}')
print(f'Non reliable point lower: {non_reliable_point}\n'
      f'Reliable point upper: {reliable_point}')


1
Reliable count: 202
Non reliable count: 9798
Reliable ratio to total: 0.0202
2
Reliable count: 202
Non reliable count: 19798
Reliable ratio to total: 0.0101
3
Reliable count: 224
Non reliable count: 29776
Reliable ratio to total: 0.007466666666666667
4
Reliable count: 271
Non reliable count: 39729
Reliable ratio to total: 0.006775
5
Reliable count: 273
Non reliable count: 49727
Reliable ratio to total: 0.00546
6
Reliable count: 273
Non reliable count: 59727
Reliable ratio to total: 0.00455
7
Reliable count: 273
Non reliable count: 69727
Reliable ratio to total: 0.0039
8
Reliable count: 274
Non reliable count: 79726
Reliable ratio to total: 0.003425
9
Reliable count: 284
Non reliable count: 89716
Reliable ratio to total: 0.0031555555555555555
10
Reliable count: 289
Non reliable count: 99711
Reliable ratio to total: 0.00289
11
Reliable count: 293
Non reliable count: 109707
Reliable ratio to total: 0.0026636363636363637
12
Reliable count: 370
Non reliable count: 119630
Reliable ratio to

In [7]:
print(f'Mainly non reliable articles: article 0-{non_reliable_point*10000}\n'
      f'Mainly reliable articles: article {reliable_point*10000}-')

Mainly non reliable articles: article 0-6650000
Mainly reliable articles: article 6660000-


In [12]:
StartPath = Path.cwd().parents[0]
_FILEPATH = StartPath / "data" / "preprocessed_dataset.csv"
_CHUNKSIZE = 50000
_OUTPUT_PATH = StartPath / "data" / "vocabulary.csv"

with pd.read_csv(
    _FILEPATH,
    chunksize=_CHUNKSIZE,
    quotechar='"',
    usecols=['content_clean'],
    low_memory=False
) as reader:
    vocab = set()
    i = 0
    for chunk in reader:
        i += 1
        print(i)
        chunk_vocab = chunk['content_clean'].str.lstrip("['").str.rstrip("']").str.split(r"', '").explode(ignore_index=True)

        vocab.update(chunk_vocab)
pd.Series(list(vocab)).to_csv(_OUTPUT_PATH, mode='w', header=False, index=False)

print("Vocabulary finished")


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
Vocabulary finished


In [4]:
StartPath = Path.cwd().parents[0]
_VOCAB_FILEPATH = StartPath / "data" / "vocabulary.csv"
_PROCESSED_FILEPATH = StartPath / "data" / "preprocessed_dataset.csv"
_OUTPUT_PATH = StartPath / "data" / "vocabulary_stats.csv"
_CHUNKSIZE = 50000

types = [
    'unreliable','hate','junksci','fake','satire','conspiracy','bias','rumor','unknown',
    'reliable','political','state','clickbait'
]
df_dict = {
    word: {
        'count': 0,
    } for word in pd.read_csv(_VOCAB_FILEPATH, header=None, low_memory=False)
}
default_row = {'count': 0}

print('constructed')


with pd.read_csv(
    _PROCESSED_FILEPATH,
    chunksize=_CHUNKSIZE,
    usecols=['content']
) as reader:
    i = 0
    for chunk in reader:
        i += 1
        print(i*_CHUNKSIZE)
        chunk_vocab = chunk['content'].str.lstrip("['").str.rstrip("']").str.split(r"', '")
        for tokens in chunk_vocab:
            if not isinstance(tokens, list):
                continue
            for token in tokens:
                df_dict.setdefault(token, default_row)
                df_dict[token]['count'] += 1

df = pd.DataFrame(df_dict).T.sort_values(by=['count'], ascending=False)
df.to_csv(_OUTPUT_PATH,mode="w",header=True)


constructed
50000
100000
150000
200000
250000
300000
350000
400000
450000
500000
550000
600000
650000
700000
750000
800000
850000
900000
950000
1000000


In [ ]:
StartPath = Path.cwd().parents[0]
_FILEPATH = StartPath / "data" / "vocabulary_stats.csv"

_OUTPUT_PATH = StartPath / "data" / "reduced_vocabulary_stats.csv"


df = pd.read_csv(_FILEPATH, low_memory=False)
df['count'] = df['count'].astype(int)
df = df.sort_values(by=['count'], ascending=False).head(10000)
df.to_csv(_OUTPUT_PATH,mode="w",header=True, index=False)

# ALL_LABELS = {'unreliable', 'hate', 'junksci', 'fake', 'satire', 'conspiracy', 'bias', 'reliable', 'political', 'state', 'clickbait'}
# df['types'] = df['types'].str.lstrip("{'").str.rstrip("}").str.split(r", '")
# df['types'] = df['types'].apply(lambda row: {key: int(val) for key, val in [label.split("': ") for label in row]})


# df['count'] = df['types'].apply(lambda r: sum([r.get(l, 0) for l in ALL_LABELS]))
# df = df.mask(df['count'] < 1).dropna()
# df = df.sort_values(by=['count'], ascending=False)


In [ ]:
_FILEPATH = StartPath / "data" / "reduced_vocabulary_stats.csv"

_OUTPUT_PATH = StartPath / "data" / "vocabulary_reliable_frequency.csv"

df = pd.read_csv(_FILEPATH, usecols=['word', 'count', 'types'], low_memory=False)
FAKE_LABELS = {'unreliable', 'hate', 'junksci', 'fake', 'satire', 'conspiracy', 'bias', 'clickbait'}
TRUE_LABELS = {'reliable', 'political', 'state'}
ALL_LABELS = {'unreliable', 'hate', 'junksci', 'fake', 'satire', 'conspiracy', 'bias', 'reliable', 'political', 'state', 'clickbait'}

df['types'] = df['types'].str.lstrip("{'").str.rstrip("}").str.split(r", '")
df['types'] = df['types'].apply(lambda row: {key: int(val) for key, val in [label.split("': ") for label in row]})

df['reliable_count'] = df['types'].apply(lambda w: sum([w.get(l, 0) for l in TRUE_LABELS]))
df['fake_count'] = df['types'].apply(lambda w: sum([w.get(l, 0) for l in FAKE_LABELS]))

df['score'] = np.log((df['reliable_count'] + 1) / (df['fake_count'] + 1))
df['score'] *= np.log1p(df['count'])

# df['reliable_freq'] = df['types'].apply(
#     lambda r:
#         sum([int(r.get(l, 0)) for l in TRUE_LABELS])/sum([int(r.get(l, 0)) for l in ALL_LABELS])
# )
# df['fake_freq'] = df['types'].apply(
#     lambda r:
#         sum([int(r.get(l, 0)) for l in FAKE_LABELS])/sum([int(r.get(l, 0)) for l in ALL_LABELS])
# )

df['normalizer'] = np.log1p(df['count'])

# df['reliable_normalized'] = df['reliable_freq']*df['normalizer']
# df['fake_normalized'] = df['fake_freq']*df['normalizer']

d_out = pd.DataFrame(
    data={
        'word': df['word'],
        'count': df['count'],
        'reliable_count': df['reliable_count'],
        'fake_count': df['fake_count'],
        # 'reliable_freq': df['reliable_freq'],
        'score': df['score'],
        'normalizer': df['normalizer']
        # 'reliable_normalized': df['reliable_normalized'],
        # 'fake_normalized': df['fake_normalized']
    }
)

d_out.to_csv(_OUTPUT_PATH, mode="w", header=True, index=False)